# WasteWiseAI — Kaggle Training & ONNX Export Pipeline
### AI Builder Congress Hackathon

This notebook trains a **YOLOv8-nano** model on a Bangladeshi canteen food dataset, evaluates performance (mAP50, mAP50-95), and exports `best.onnx` and `classes.json` for drop-in loading into the WasteWiseAI production platform.

In [ ]:
# Step 1: Install Ultralytics and ONNX dependencies
!pip install -q ultralytics onnx onnxslim onnxruntime

In [ ]:
import os
import json
import shutil
from pathlib import Path
from ultralytics import YOLO

# Configurable dataset input path on Kaggle
DATASET_INPUT = Path('/kaggle/input/bangladeshi-canteen-food-dataset')
OUTPUT_DIR = Path('/kaggle/working/export')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Dataset Path: {DATASET_INPUT}")
print(f"Output Path: {OUTPUT_DIR}")

### Step 2: Define and Validate Classes
Classes support two groups:
1. **Food Classes**: `rice`, `dal`, `bhuna_khichuri`, `chicken_curry`, `fish_curry`, `vegetable`, `egg_curry`, `bhorta`
2. **Plate States**: `empty_plate`, `partial_leftover`, `full_plate`

In [ ]:
classes = [
    "rice",
    "dal",
    "bhuna_khichuri",
    "chicken_curry",
    "fish_curry",
    "vegetable",
    "egg_curry",
    "bhorta",
    "empty_plate",
    "partial_leftover",
    "full_plate"
]

classes_json_path = OUTPUT_DIR / 'classes.json'
with open(classes_json_path, 'w') as f:
    json.dump({
        "food_classes": [c for c in classes if c not in ["empty_plate", "partial_leftover", "full_plate"]],
        "plate_state_classes": ["empty_plate", "partial_leftover", "full_plate"],
        "all_classes": classes
    }, f, indent=2)

print(f"Created {classes_json_path}")

In [ ]:
# Step 3: Write dataset.yaml
yaml_path = OUTPUT_DIR / 'dataset.yaml'
yaml_content = f"""path: {DATASET_INPUT.absolute()}
train: images/train
val: images/val
test: images/test

names:
"""
for i, name in enumerate(classes):
    yaml_content += f"  {i}: {name}\n"

with open(yaml_path, 'w') as f:
    f.write(yaml_content)

print(f"Generated dataset config at {yaml_path}")

### Step 4: Train YOLOv8n with Canteen Data Augmentation
Applies lighting variations, blur, plate rotation, and mosaic.

In [ ]:
model = YOLO('yolov8n.pt')

results = model.train(
    data=str(yaml_path),
    epochs=50,
    batch=16,
    imgsz=640,
    patience=10,
    hsv_h=0.015,
    hsv_s=0.6,
    hsv_v=0.4,
    degrees=15.0,
    translate=0.1,
    scale=0.5,
    mosaic=1.0,
    mixup=0.1,
    project=str(OUTPUT_DIR / 'runs'),
    name='wastewise_yolov8n'
)

### Step 5: Evaluate Model Performance (mAP50 & mAP50-95)

In [ ]:
metrics = model.val()
print(f"Validation mAP50:    {metrics.box.map50:.4f}")
print(f"Validation mAP50-95: {metrics.box.map:.4f}")

### Step 6: Export to CPU-Optimized ONNX Format
Exports `best.onnx` with 640px static input size for fast CPU inference under 400ms on Render.

In [ ]:
best_pt = OUTPUT_DIR / 'runs' / 'wastewise_yolov8n' / 'weights' / 'best.pt'
best_model = YOLO(str(best_pt))

exported_onnx = best_model.export(
    format='onnx',
    imgsz=640,
    dynamic=False,
    simplify=True,
    opset=12
)

shutil.copy(exported_onnx, OUTPUT_DIR / 'best.onnx')
print(f"Successfully exported {OUTPUT_DIR / 'best.onnx'}")
print("Download best.onnx and classes.json and place into WasteWiseAI/models/")